In [8]:
import pandas as pd
import sqlite3
from pathlib import Path
import numpy as np

project_root = Path.cwd().parent
db_path = project_root / "data" / "cloudtrail.db"

assert db_path.exists()

connection = sqlite3.connect(db_path)

In [2]:
df_counts = pd.read_sql_query(
    """
    SELECT
        user,
        COUNT(*) AS event_count
    FROM events
    GROUP BY user
    ORDER BY user
    """,
    connection
)

df_counts

,user,event_count
0,admin,36
1,analyst,18
2,developer,22
3,service-user,19


In [13]:
event_counts = df_counts["event_count"]
print("Mean of event counts:" , event_counts.mean())
print("Median of event counts:", event_counts.median())
print("Variance of event counts:" , event_counts.var())
print("Quantile of event counts:", event_counts.quantile([0, 0.25, 0.50, 0.75, 1]))

Mean of event counts: 23.75
Median of event counts: 20.5
Variance of event counts: 69.58333333333333
Quantile of event counts: 0.00    18.00
0.25    18.75
0.50    20.50
0.75    25.50
1.00    36.00
Name: event_count, dtype: float64


In [14]:
# Outlier Experiment
event_counts_outlier = pd.concat(
    [
        event_counts,
        pd.Series([500])
    ],
    ignore_index=True
)

In [19]:
comparison_df = pd.DataFrame({
    "Original": [
        event_counts.mean(),
        event_counts.var(),
        event_counts.quantile(0.25),
        event_counts.median(),
        event_counts.quantile(0.75)
    ],
    "With Outlier":[
        event_counts_outlier.mean(),
        event_counts_outlier.var(),
        event_counts_outlier.quantile(0.25),
        event_counts_outlier.median(),
        event_counts_outlier.quantile(0.75)
    ]
}, index=[
    "Mean",
    "Variance",
    "25th percentile",
    "Median",
    "75th percentile"
])

comparison_df

,Original,With Outlier
Mean,23.750000,119.0
Variance,69.583333,45415.0
25th percentile,18.750000,19.0
Median,20.500000,22.0
75th percentile,25.500000,36.0


### Day 22 — Descriptive Statistics: Averages and Spread

Analyzed event counts across four synthetic CloudTrail identities to understand measures of central tendency and variability.

**Key accomplishments:**
- Calculated the mean, median, sample variance, and 25th, 50th, and 75th percentiles using pandas.
- Introduced a hypothetical identity generating 500 events to simulate an extreme outlier.
- Created a comparison table showing how descriptive statistics change when an outlier is introduced.
- Examined the difference between typical activity, average activity, and total workload.

**Results:**
- Original mean: 23.75 events per identity.
- Original median: 20.5 events per identity.
- Mean with outlier: 119 events per identity.
- Median with outlier: 22 events per identity.

**Key takeaways:**
- The mean is sensitive to extreme values because it incorporates the magnitude of every observation.
- The median is more resistant to outliers and can better represent typical behavior in skewed data.
- Variance is particularly sensitive to outliers because deviations from the mean are squared.
- High event volume does not automatically indicate suspicious or malicious behavior.

**Limitations:** The original dataset contains only four synthetic identities. The 500-event observation was artificially introduced, and these results cannot establish real-world security behavior.

**Reflection:** I learned that choosing a statistical measure depends on the question being asked. The median can describe typical activity, while the mean and total event count answer different operational questions.